In [ ]:
import glob
import warnings
warnings.filterwarnings("ignore")
import pandas as pd

OUT_PATH = "submission.csv"
RAUNAK_TOKENS = ["aleixlopez", "birdclef-2026-ported-raunak-v6-v1"]
MEENAL_TOKENS = ["aleixlopez", "birdclef-2026-meenal-v1"]
EOS6_TOKENS = ["birdclef-2026-eos-9"]
WEIGHTS = {
    "raunak": 0.40,
    "meenal": 0.40,
    "eos6": 0.20,
}


def _find_submission(required_tokens):
    # Search broadly: kernel outputs may be at /kaggle/input/{slug}/ or /kaggle/input/notebooks/{user}/{slug}/
    candidates = (
        glob.glob("/kaggle/input/**/submission.csv", recursive=True) +
        glob.glob("/kaggle/working/**/submission.csv", recursive=True)
    )
    matches = []
    for path in candidates:
        normalized = path.lower().replace("\\", "/")
        if all(token.lower() in normalized for token in required_tokens):
            matches.append(path)
    if not matches:
        all_csvs = glob.glob("/kaggle/input/**/submission.csv", recursive=True)
        raise FileNotFoundError(
            f"Could not find submission.csv matching tokens={required_tokens}\n"
            f"Available submission.csv files:\n" + "\n".join(f"  {p}" for p in sorted(all_csvs))
        )
    return sorted(matches)[-1]


def _find_sample_submission():
    candidates = glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True)
    if not candidates:
        raise FileNotFoundError("sample_submission.csv not found under /kaggle/input")
    return sorted(candidates)[0]


def _align_to_template(df, row_ids, class_cols, source_name):
    if "row_id" not in df.columns:
        raise ValueError(f"{source_name}: source submission is missing row_id")
    missing_cols = [c for c in class_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"{source_name}: source submission missing class columns: {missing_cols[:5]}")
    source = df[["row_id"] + class_cols].copy()
    source["row_id"] = source["row_id"].astype(str).str.strip()
    source = source.drop_duplicates(subset=["row_id"]).set_index("row_id")
    template_index = pd.Index(row_ids)
    missing_rows = int((~template_index.isin(source.index)).sum())
    if missing_rows:
        print(f"WARNING: {source_name}: {missing_rows} missing row_ids — filling with uniform baseline")
    aligned = source.reindex(row_ids)
    # Fill any missing rows with uniform probability (1/n_classes)
    if aligned.isna().any().any():
        fill_val = 1.0 / len(class_cols)
        aligned = aligned.fillna(fill_val)
    return aligned


raunak_path = _find_submission(RAUNAK_TOKENS)
meenal_path = _find_submission(MEENAL_TOKENS)
eos6_path = _find_submission(EOS6_TOKENS)
sample_path = _find_sample_submission()
print("Using Raunak submission:", raunak_path)
print("Using Meenal submission:", meenal_path)
print("Using EoS6 submission:", eos6_path)
print("Using sample submission template:", sample_path)

raunak = pd.read_csv(raunak_path)
meenal = pd.read_csv(meenal_path)
eos6 = pd.read_csv(eos6_path)
sample = pd.read_csv(sample_path)

if "row_id" not in sample.columns:
    raise ValueError("sample_submission.csv missing row_id")
sample["row_id"] = sample["row_id"].astype(str).str.strip()
class_cols = [c for c in sample.columns if c != "row_id"]
row_ids = sample["row_id"].tolist()

raunak_aligned = _align_to_template(raunak, row_ids, class_cols, "Raunak")
meenal_aligned = _align_to_template(meenal, row_ids, class_cols, "Meenal")
eos6_aligned = _align_to_template(eos6, row_ids, class_cols, "EoS6")

merged = sample[["row_id"]].copy()
for c in class_cols:
    rank_raunak = raunak_aligned[c].rank(method="average", pct=True)
    rank_meenal = meenal_aligned[c].rank(method="average", pct=True)
    rank_eos6 = eos6_aligned[c].rank(method="average", pct=True)
    merged[c] = (
        WEIGHTS["raunak"] * rank_raunak
        + WEIGHTS["meenal"] * rank_meenal
        + WEIGHTS["eos6"] * rank_eos6
    )

merged = merged[["row_id"] + class_cols]
if merged[class_cols].isna().any().any():
    raise ValueError("Blend produced NaNs; aborting submission generation")

# ── Site/Hour Prior Calibration ────────────────────────────────────────────
# Applies a Bayesian-smoothed site+hour species prior from labeled soundscapes.
# Multiplicative in log-space: log_p_adjusted = log_p_model + lambda * log_prior_relative
# Safe: Bayesian smoothing falls back to global prior when data is sparse.

import re
import numpy as np

PRIOR_LAMBDA  = 0.30   # strength of prior adjustment (tune: 0.0 = off, 0.5 = strong)
PRIOR_SMOOTH  = 8.0    # pseudo-count for Bayesian smoothing (higher = more conservative)
SS_LABELS_CSV = "/kaggle/input/birdclef-2026/train_soundscapes_labels.csv"

FNAME_RE  = re.compile(r"BC2026_(?:Train|Test)_\d+_(S\d+)_\d{8}_(\d{6})\.ogg")
ROW_ID_RE = re.compile(r"BC2026_(?:Train|Test)_\d+_(S\d+)_\d{8}_(\d{6})_\d+")

def _parse_row_id(row_id):
    m = ROW_ID_RE.match(str(row_id))
    if not m:
        return None, -1
    return m.group(1), int(m.group(2)[:2])

def _build_priors(ss_labels_path, class_cols):
    eps = 1e-6
    n   = len(class_cols)
    ci  = {c: i for i, c in enumerate(class_cols)}

    try:
        ss = pd.read_csv(ss_labels_path)
    except FileNotFoundError:
        print(f"[prior] WARNING: {ss_labels_path} not found — skipping prior calibration")
        return None

    rows = []
    for _, r in ss.iterrows():
        m = FNAME_RE.match(str(r.get("filename", "")))
        if m and str(r.get("primary_label","")) in ci:
            rows.append({"site": m.group(1), "hour": int(m.group(2)[:2]), "cls": ci[str(r["primary_label"])]})

    if not rows:
        print("[prior] WARNING: no matching rows in soundscape labels — skipping")
        return None

    df = pd.DataFrame(rows)
    print(f"[prior] Built from {len(df)} annotations ({df['site'].nunique()} sites, "
          f"{df['hour'].nunique()} hours, {df['cls'].nunique()} species)")

    # Global prior (base rate per class)
    global_cnt = np.bincount(df["cls"].values, minlength=n).astype(np.float32)
    global_p   = (global_cnt + eps) / (global_cnt.sum() + n * eps)

    # Site prior
    site_p, site_n = {}, {}
    for site, grp in df.groupby("site"):
        cnt = np.bincount(grp["cls"].values, minlength=n).astype(np.float32)
        site_n[site] = cnt.sum()
        site_p[site] = (cnt + eps) / (cnt.sum() + n * eps)

    # Hour prior
    hour_p, hour_n = {}, {}
    for hour, grp in df.groupby("hour"):
        cnt = np.bincount(grp["cls"].values, minlength=n).astype(np.float32)
        hour_n[int(hour)] = cnt.sum()
        hour_p[int(hour)] = (cnt + eps) / (cnt.sum() + n * eps)

    return {"global_p": global_p, "site_p": site_p, "site_n": site_n,
            "hour_p": hour_p, "hour_n": hour_n}

def _apply_prior(scores_arr, row_ids, tables, lam=PRIOR_LAMBDA, smooth=PRIOR_SMOOTH):
    """Bayesian-smoothed multiplicative prior. scores_arr shape: (N, C) probabilities.
    Multi-label safe: adjusts each class independently, no sum-to-1 renorm."""
    if tables is None or lam == 0.0:
        return scores_arr

    eps    = 1e-7
    result = scores_arr.copy().astype(np.float32)
    g_p    = tables["global_p"]  # (C,) global species prevalence

    for i, rid in enumerate(row_ids):
        site, hour = _parse_row_id(rid)

        # Start from global prior
        p_adj = g_p.copy()

        # Bayesian-smoothed site prior: w = n / (n + smooth)
        if site and site in tables["site_p"]:
            w = tables["site_n"][site] / (tables["site_n"][site] + smooth)
            p_adj = w * tables["site_p"][site] + (1.0 - w) * p_adj

        # Bayesian-smoothed hour prior
        if hour >= 0 and hour in tables["hour_p"]:
            w = tables["hour_n"][hour] / (tables["hour_n"][hour] + smooth)
            p_adj = w * tables["hour_p"][hour] + (1.0 - w) * p_adj

        # Multiplicative boost: species more common here get a boost, rare get dampened
        # ratio is bounded to prevent extreme adjustments
        ratio = np.clip(p_adj / (g_p + eps), 0.5, 2.0)
        result[i] = np.clip(result[i] * (ratio ** lam), 0.0, 1.0)

    return result


prior_tables = _build_priors(SS_LABELS_CSV, class_cols)

blend_arr = merged[class_cols].values.astype(np.float32)
blend_arr = _apply_prior(blend_arr, merged["row_id"].tolist(), prior_tables)
merged[class_cols] = blend_arr

merged.to_csv(OUT_PATH, index=False, float_format="%.10f")
print(merged.shape)
print(f"[prior] Wrote {OUT_PATH} with prior calibration (lambda={PRIOR_LAMBDA})")
